# Experiment 005: 0.5B base-to-GRPO replication, seeds 1 and 2 (Colab)

Two base-start GRPO runs under the unchanged frozen values,
identical to the seed-0 cell except the seed and the registered ladder
switch. Both are prospectively fixed exploratory arms of the adaptive
extension and their configs carry that status. Registered question this
stage answers: is the seed-0 format collapse (fmt .018 without the warm
start) systematic or an accident of one draw?

Runs on the A100-40GB, matching its seed 0; the
GPU cell REJECTS an 80GB allocation, release the runtime and re-roll until
a 40GB card appears. Base evals are reused from the shared 005 directory.

~11.7h total at the cheaper 40GB rate, resume-safe.
If a run OOMs: archive BOTH its directories, results/<run_tag> and
checkpoints/<run_tag>, into failed_attempts per the registered procedure;
no retry at changed settings.

## 0. Compute units and mount Drive

Fill `cu_before` and `unit_rate` now.

In [1]:
# note the balance before anything burns units, then save it after drive mounts
import glob, json, os, time
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = 354.93     # fill from the resources panel
unit_rate = 5.3     # units per hour
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

# close unfinished session with the balance captured at this runtime start
old_sessions = [p for p in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not p.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None:
    print('reminder: fill cu_before, then rerun this cell to save the session')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])

    if not os.path.exists(open_path):
        start = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                 'notebook': '005_colab_0p5b_base_grpo_seeds12', 'colab_cu_before': cu_before, 'unit_rate': unit_rate}
        with open(open_path, 'w') as f:
            json.dump(start, f, indent=2)
        print('saved open session:', session_id)
print('drive mounted ->', ROOT)


session 20260804_103526 | cu_before = 354.93
Mounted at /content/drive
closed previous session: 20260803_140948
saved open session: 20260804_103526
drive mounted -> /content/drive/MyDrive/inm363


## 1. GPU check: A100 with 40GB required

In [2]:
import subprocess, torch
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print(gpu.strip() or 'no gpu visible')
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'total vram: {total_gb:.1f} GiB')
# I pin this cell to the 40gb variant its seed 0 used, so an 80gb
# allocation is rejected even though it would fit
assert 'A100' in gpu, 'need an a100 runtime'
assert 39 <= total_gb < 45, f'need the a100 40gb variant per the 6z rule, got {total_gb:.0f} GiB, re-roll the runtime'

GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-bbf3350f-933d-257a-b67b-c7b7ef7e78e1)
total vram: 39.5 GiB


## 2. Clone or update the repo (automatic)

In [3]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    # a resumed runtime already has the clone, in that case pull instead of failing
    # and silently running whatever old code is lying around
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())

Cloning into 'INM363-IndividualProject-Dissertation'...
remote: Enumerating objects: 3310, done.
remote: Counting objects: 100% (794/794), done.
remote: Compressing objects: 100% (539/539), done.
remote: Total 3310 (delta 490), reused 555 (delta 255), pack-reused 2516 (from 1)
Receiving objects: 100% (3310/3310), 98.53 MiB | 43.02 MiB/s, done.
Resolving deltas: 100% (2109/2109), done.
Updating files: 100% (1489/1489), done.
working in: /content/INM363-IndividualProject-Dissertation/001_Code


## 3. Install dependencies

In [4]:
!pip install -q trl==1.7.1 transformers==5.5.4 peft==0.19.1 accelerate==1.14.0 datasets==4.8.4 wandb pyyaml
# colab preinstalls a torchao that peft 0.19.1 chokes on inside its lora
# dispatcher. No quantization is used here, so just drop it
!pip uninstall -q -y torchao

# pin the runs to what the seed-0 manifest actually records
import importlib, json, os, subprocess, yaml
ref_path = 'results/005_COLAB_FULL_B05_base_grpo_seed0/manifest.json'
assert os.path.exists(ref_path), 'reference manifest missing, pull the repo with results before training'
ref = json.load(open(ref_path))
for lib in ('torch', 'trl', 'transformers', 'peft', 'accelerate'):
    have = importlib.import_module(lib).__version__
    assert have == ref['versions'][lib], f'{lib} {have} differs from the recorded {ref["versions"][lib]}'

# the manifest predates the dataset-revision fields, so the pins are checked
# against the audited revisions instead
from data.mmlu import MMLU_REVISION
from data.gsm8k import GSM8K_REVISION
assert MMLU_REVISION == 'c30699e8356da336a370243923dbaf21066bb9fe'
assert GSM8K_REVISION == '740312add88f781978c0658806c59bc2815b9866'

# the seed-0 config values must still equal the recorded run config
commit = ref['git']['commit']
assert ref['git']['dirty'] is False
old = subprocess.run(['git', 'show', f"{commit}:001_Code/configs/005_colab_full_b05_base_grpo_seed0.yaml"],
                     capture_output=True, text=True)
if old.returncode != 0:
    old = subprocess.run(['git', 'show', f"{commit}:configs/005_colab_full_b05_base_grpo_seed0.yaml"],
                         capture_output=True, text=True)
assert old.returncode == 0, 'could not read the seed-0 config at its run commit'
recorded_cfg = yaml.safe_load(old.stdout)
current_cfg = yaml.safe_load(open('configs/005_colab_full_b05_base_grpo_seed0.yaml'))
assert recorded_cfg == current_cfg, 'seed-0 config values drifted since its recorded run'
print('environment, dataset pins and historical config values verified')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 132.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 53.7 MB/s eta 0:00:00
environment, dataset pins and historical config values verified


## 4. W&B key (automatic)

Reminder: colab secrets are granted per notebook, switch on access for
`WANDB_API_KEY` under the key icon before running. This key is only used for experiment tracking.


In [5]:
import getpass
import os

wkey = None
try:
    from google.colab import userdata
    wkey = userdata.get('WANDB_API_KEY')
except Exception:
    wkey = None

if not wkey:
    wkey = getpass.getpass('W&B API key: ').strip()
if not wkey:
    raise RuntimeError('WANDB_API_KEY is required')

os.environ['WANDB_API_KEY'] = wkey
with open('wandb_api_key.txt', 'w') as f:
    f.write(wkey.strip())
print('W&B key loaded:', os.path.exists('wandb_api_key.txt'))


W&B key loaded: True


## 5. Check the Drive record and free space

In [6]:
import os, shutil
if os.path.exists(open_path):
    print('open compute-unit record:', open_path)
else:
    print('open record missing, rerun section 0 with cu_before filled')
free_gb = shutil.disk_usage('/content/drive/MyDrive').free / 1024**3
print(f'drive free: {free_gb:.0f} GB')
assert free_gb >= 10, 'need >=10GB free on drive for the 0.5b runs'

open compute-unit record: /content/drive/MyDrive/inm363/results/open_cu_sessions/20260804_103526.json
drive free: 179 GB


## 6. Build the data splits

In [7]:
!python scripts/build_data.py


README.md: 100% 7.93k/7.93k [00:00<00:00, 16.7MB/s]

main/train-00000-of-00001.parquet: downloading bytes:  30% 688k/2.31M [00:00<00:01, 1.46MB/s]
main/train-00000-of-00001.parquet: downloading bytes: 100% 2.30M/2.30M [00:00<00:00, 3.95MB/s,  227kB/s  ]
main/train-00000-of-00001.parquet: reconstructing file: 100% 2.31M/2.31M [00:00<00:00, 3.96MB/s,  228kB/s  ]

main/test-00000-of-00001.parquet: downloading bytes:   0% 0.00/419k [00:00<?, ?B/s]
main/test-00000-of-00001.parquet: downloading bytes: 100% 419k/419k [00:00<00:00, 1.11MB/s, 41.7kB/s  ]
main/test-00000-of-00001.parquet: reconstructing file: 100% 419k/419k [00:00<00:00, 1.11MB/s, 41.7kB/s  ]
Generating train split: 100% 7473/7473 [00:00<00:00, 376048.68 examples/s]
Generating test split: 100% 1319/1319 [00:00<00:00, 197786.53 examples/s]
README.md: 100% 53.2k/53.2k [00:00<00:00, 98.9MB/s]
dataset_infos.json: 100% 138k/138k [00:00<00:00, 161MB/s]

high_school_psychology/test-00000-of-000(…): downloading bytes:  18% 16.4k/92.8k [

## 7. Preflight: the shared 005 base evals must exist and be complete

In [8]:
import json, os
D = '/content/drive/MyDrive/inm363'
be = os.path.join(D, 'results', 'base_evals', '005_COLAB_FULL_B05')
# the runner skips base evals when these exist, so their completeness is
# checked here rather than discovered after the expensive runs
for f in ('base_eval.json', 'base_logprob.json'):
    fp = os.path.join(be, f)
    assert os.path.exists(fp), f'shared 005 base eval missing: {f}'
    assert json.load(open(fp))['n_items'] == 1884, f'{f}: wrong item count'
for f in ('base_preds.jsonl', 'base_logprob_preds.jsonl'):
    fp = os.path.join(be, f)
    assert os.path.exists(fp), f'shared 005 base eval missing: {f}'
    rows = sum(1 for line in open(fp) if line.strip())
    assert rows == 1884, f'{f}: {rows} rows'
print('shared base evals present and complete, both runs will skip them')

shared base evals present and complete, both runs will skip them


## 8. Wiring check

In [9]:
import subprocess, sys
for s in (1, 2):
    print('wiring: seed', s)
    subprocess.run([sys.executable, 'scripts/run.py', '--config',
                    f'configs/005_colab_full_b05_base_grpo_seed{s}.yaml'], check=True)

wiring: seed 1
wiring: seed 2


## 9. The run

**If colab disconnects:** re-run this cell, finished stages skip.

In [ ]:
# both cost fields must be filled before the long runs start
assert cu_before is not None and unit_rate is not None, 'fill cu_before and unit_rate in the units cell'
# python loop with ! calls so the logs stream live, exit code checked so a
# failed run stops the session instead of hiding behind the next one
for cfg in ('005_colab_full_b05_base_grpo_seed1', '005_colab_full_b05_base_grpo_seed2'):
    print(f'===== {cfg} =====')
    !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python scripts/run.py --config configs/{cfg}.yaml --stage all
    if _exit_code != 0:
        raise SystemExit(f'{cfg} failed with exit code {_exit_code}; an oom means archive results/<run_tag> AND checkpoints/<run_tag> per the registered procedure, never retry with changed settings')

Streaming output truncated to the last 5000 lines.
│ │               │ to help with  │               │              │           │ │
│ │               │ the creation  │               │              │           │ │
│ │               │ of            │               │              │           │ │
│ │               │ cooperative   │               │              │           │ │
│ │               │ teams.        │               │              │           │ │
│ │               │               │               │              │           │ │
│ │               │ Consider the  │               │              │           │ │
│ │               │ context loop! │               │              │           │ │
│ │               │ Planning is   │               │              │           │ │
│ │               │ the point of  │               │              │           │ │
│ │               │ both timing   │               │              │           │ │
│ │               │ and assigning │               │       

## 10. Peek at the results

In [ ]:
import json, os
D = '/content/drive/MyDrive/inm363'
for tag in ('005_COLAB_FULL_B05_base_grpo_seed0', '005_COLAB_FULL_B05_base_grpo_seed1',
            '005_COLAB_FULL_B05_base_grpo_seed2'):
    sm = os.path.join(D, 'results', tag, 'summary.json')
    if os.path.exists(sm):
        j = json.load(open(sm))
        print(f"{tag}: grpo {j['grpo_acc']}")
    else:
        print(f'{tag}: no summary yet')

005_COLAB_FULL_B05_base_grpo_seed0: grpo {'easy': 0.48236632536973834, 'medium': 0.3727422003284072, 'hard': 0.255050505050505}
005_COLAB_FULL_B05_base_grpo_seed1: grpo {'easy': 0.4664391353811149, 'medium': 0.3333333333333333, 'hard': 0.2676767676767677}
005_COLAB_FULL_B05_base_grpo_seed2: grpo {'easy': 0.5073947667804323, 'medium': 0.3924466338259442, 'hard': 0.26515151515151514}


## 11. Close the units record

In [ ]:
# closing balance, completes the open record for this session
import json, os, time
cu_after = 292.16      # fill from the resources panel again
if cu_after is None:
    print('reminder: fill cu_after for the record')
elif not os.path.isdir('/content/drive/MyDrive'):
    print('drive not mounted, run the mount cell first')
else:
    os.makedirs(OPEN_UNITS, exist_ok=True)
    open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
    if not os.path.exists(open_path):
        print('no open record for this session, it may already be closed')
    else:
        with open(open_path) as f:
            rec = json.load(f)
        rec['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        rec['colab_cu_after'] = cu_after
        rec['colab_cu_used'] = round(rec['colab_cu_before'] - cu_after, 2)
        rec['closed_on_resume'] = False
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(rec) + '\n')
        os.remove(open_path)
        print('logged:', rec)

reminder: fill cu_after for the record
